# Glaucoma v6 FINAL protocol: 10 grouped folds x 5 seeds (one arm per run)
**Rule (locked before running, see project note `final_selection_rule_v6`):** final configuration = ODIR-pretrained EfficientNet-B0 224 (three independent pretraining runs: seeds 42, 7, 123) + EfficientNet-B0 384 (ImageNet). The pre-meeting ResNet50 recipe is run in the same protocol as the baseline. The test set is **not used**. Whatever comes out is what gets reported.

**One arm per Kaggle session.** Set `ARM` in the next cell to one of: `odir42`, `odir7`, `odir123`, `eff384`, `resnet50`. Run all five arms (any order), downloading each zip right after it finishes.

| ARM | what it is | rough time (T4) |
|---|---|---|
| `odir42` | EffNet-B0 224 from the saved ODIR seed-42 backbone | about 1.4 h |
| `odir7` | EffNet-B0 224 from the saved ODIR seed-7 backbone | about 1.4 h |
| `odir123` | NEW third ODIR pretraining (seed 123), then EffNet-B0 224 | about 1.8 h |
| `eff384` | EffNet-B0 384, ImageNet weights | about 3.4 h |
| `resnet50` | ResNet50 224, ImageNet weights (baseline) | about 1.4 h |

**Attach these inputs:**
- `odir42`, `odir7`, `resnet50`: the dataset with **`glaucoma_pack_256.tar`** + the **v6 bundle** dataset.
- `eff384`: the dataset with **`glaucoma_pack_440.tar`** + the **v6 bundle**.
- `odir123`: the `glaucoma_pack_256.tar` dataset + the **v6 bundle** + the public ODIR dataset `andrewmvd/ocular-disease-recognition-odir5k`.

(The packs are single `.tar` files on purpose: Kaggle loads one big file much faster than 2,000 small images. The notebook unpacks the tar into `/tmp` in a few seconds. If a folder-style `pack/` dataset is attached instead, that still works.)

Kaggle settings: **GPU T4, Internet On**. Session options: **Persistence = Files only**. Use an **interactive session** (not Save & Run All: long commits hung before). Keep the tab open and click *Continue editing* if asked "Are you still there?". The run is resumable: finished (fold, seed) runs are skipped if you restart in the same session.

First do one `QUICK_TEST = True` run (a few minutes) to check everything is attached, then set it to False.

In [ ]:
# ---- Settings ----
ARM = "odir42"          # odir42 | odir7 | odir123 | eff384 | resnet50
QUICK_TEST = True       # True = tiny smoke test (a few minutes). Set False for the real run.
USE_AMP = True
N_FOLDS = 10
SEEDS = (101, 202, 303, 404, 505)      # fixed in advance
assert ARM in ("odir42", "odir7", "odir123", "eff384", "resnet50")

In [ ]:
# ---- Find code, packs, backbones ----
import sys, glob, os
from pathlib import Path
hits = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
assert hits, 'v6 bundle not found: is the v6 bundle dataset attached?'
CODE_DIR = Path(hits[0]).parents[1]
sys.path.insert(0, str(CODE_DIR))
WORK = Path('/kaggle/working')
import torch
from src.config import find_pack_dir
print('code:', CODE_DIR, '| CUDA:', torch.cuda.is_available())
BACKBONES = {'odir42': CODE_DIR / 'backbones' / 'odir_binary_effb0.pt',
             'odir7':  CODE_DIR / 'backbones' / 'odir_binary_effb0_s7.pt'}
if ARM in BACKBONES:
    assert BACKBONES[ARM].exists(), f'backbone file missing: {BACKBONES[ARM]}'
print('ARM =', ARM, '| folds =', N_FOLDS, '| seeds =', SEEDS, '| quick =', QUICK_TEST)

# ---- Locate the image pack: prefer a single .tar (fast to load), fall back to a folder-style dataset ----
import tarfile
def locate_pack(name):
    tars = glob.glob(f'/kaggle/input/**/glaucoma_{name}.tar', recursive=True)
    if tars:
        dest = Path('/tmp/glaucoma_data')
        if not (dest / name / 'manifest.csv').exists():
            print('unpacking', tars[0], '...')
            with tarfile.open(tars[0]) as t:
                t.extractall(dest)
        n = len(list((dest / name / 'images').glob('*.png')))
        print(f'{name}: {n} images unpacked at {dest / name}')
        assert n == 2099, f'expected 2099 images, found {n}'
        return dest / name
    return find_pack_dir(name)
PACK_NAME = 'pack_440' if ARM == 'eff384' else 'pack'
PACK_DIR = locate_pack(PACK_NAME)
print('pack folder:', PACK_DIR)

In [ ]:
# ---- Run ----
import zipfile
from src.config import Config
from src.cv import run_cv
cfg = None
OUT = WORK / f'final10x5_{ARM}' ; OUT = Path(str(OUT) + ('_quick' if QUICK_TEST else ''))
common = dict(pack_dir=PACK_DIR, quick_test=QUICK_TEST, use_amp=USE_AMP, save_checkpoints=False, n_folds=N_FOLDS, seeds=SEEDS, out_dir=OUT)
try:
    if ARM == 'resnet50':
        cfg = Config(arch='resnet50', res=224, **common)
    elif ARM == 'eff384':
        cfg = Config(arch='efficientnet_b0', res=384, **common)
    elif ARM in ('odir42', 'odir7'):
        cfg = Config(arch='efficientnet_b0', res=224, init_weights=BACKBONES[ARM], init_tag='odirbin',
                     pretrain_seed=int(ARM.replace('odir', '')), **common)
    else:  # odir123: new ODIR pretraining first (same code and settings as the seed-42 and seed-7 runs)
        from src.odir import prepare_odir
        from src.pretrain import pretrain
        hits = glob.glob('/kaggle/input/**/full_df.csv', recursive=True) + glob.glob('/kaggle/input/**/data.csv', recursive=True)
        hits = sorted(hits, key=lambda p: ('odir' not in p.lower(), 'full_df' not in p))
        assert hits, 'ODIR csv not found: is the ODIR dataset attached?'
        ODIR_ROOT = Path(hits[0]).parent
        PACK = PACK_DIR; TARGET_HASHES = CODE_DIR / 'target_hashes.csv'
        assert TARGET_HASHES.exists() and TARGET_HASHES.with_name('test_thumbs_128.npz').exists(), 'target_hashes.csv / test_thumbs_128.npz missing from the bundle'
        ODIR_OUT, ODIR_IMG = WORK / 'odir_out', WORK / 'odir_pack'
        prep = prepare_odir(ODIR_ROOT, TARGET_HASHES, PACK, ODIR_OUT, ODIR_IMG, quick_n=60 if QUICK_TEST else None, arm='binary')
        if prep['test_matches']:
            raise RuntimeError('ODIR/TEST overlap found - see odir_dedupe_audit.csv. Not continuing.')
        pt, odir_metrics = pretrain('binary', ODIR_OUT / 'odir_manifest.csv', ODIR_IMG / 'images', ODIR_OUT,
                                    quick=QUICK_TEST, amp=USE_AMP, seed=123)
        cfg = Config(arch='efficientnet_b0', res=224, init_weights=pt, init_tag='odirbin', pretrain_seed=123, **common)
    print('output folder:', cfg.out_dir, '| n_folds:', cfg.n_folds, '| seeds:', cfg.seeds)
    ok = run_cv(cfg)
    print('ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED - read the log above')
finally:
    zpath = WORK / f"final10x5_{ARM}{'_quick' if QUICK_TEST else ''}_outputs.zip"
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        folders = ([cfg.out_dir] if cfg else []) + ([WORK / 'odir_out'] if ARM == 'odir123' and (WORK / 'odir_out').exists() else [])
        for folder in folders:
            for f in Path(folder).rglob('*'):
                if f.is_file():
                    z.write(f, Path(folder).name + '/' + str(f.relative_to(folder)))
    print('\n' + '=' * 70 + '\nZip created:', zpath, f'({zpath.stat().st_size/1e6:.1f} MB)')
    print('DOWNLOAD THE ZIP NOW (Output tab); the session wipes /kaggle/working when it stops')
    print('=' * 70)

In [ ]:
# ---- Results ----
import json
from IPython.display import Image, display
res = json.load(open(cfg.out_dir / 'results.json'))
print(json.dumps({k: res[k] for k in ['init', 'pretrain_seed', 'arch', 'img_size', 'n_folds', 'seeds', 'mean_std_across_folds', 'pooled_out_of_fold']}, indent=2))
for f in ['fig_pooled_roc_pr.png', 'fig_per_fold_auc.png']:
    display(Image(filename=str(cfg.out_dir / f)))
print('\nREMINDER: DOWNLOAD THE ZIP NOW (Output tab).')